In [1]:
import docling
print("Docling installed successfully!")

Docling installed successfully!


In [2]:
from docling.document_converter import DocumentConverter

C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
converter = DocumentConverter()

In [4]:
pdf_path = "C:/Users/Administrator/Desktop/AI-Portfolio/RAG/data/MRM/SR-11-7_ModelRiskMgmt_2011.pdf"

In [6]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()

# SR11-7 is a digital PDF
pipeline_options.do_ocr = False

# Keep table extraction enabled
pipeline_options.do_table_structure = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

result = converter.convert(pdf_path)

print("Success!")

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2024.48it/s]


Success!


In [7]:
doc= result.document

In [8]:
markdown = doc.export_to_markdown()
print(markdown[:3000])

April 4, 2011

## CONTENTS

| I. Introduction, page 1                                |
|--------------------------------------------------------|
| II. Purpose and Scope, page 2                          |
| III. Overview of Model Risk Management, page 3         |
| IV. Model Development, Implementation, and Use, page 5 |
| V. Model Validation, page 9                            |
| VI. Governance, Policies, and Controls, page 16        |
| VII. Conclusion, page 21                               |

## I.  INTRODUCTION

Banks rely heavily  on quantitative  analysis and models in most aspects of financial decision making. [Fotnote 1 They routinely use models for a broad range of activities,  including underwriting credits; valuing exposures, instruments,  and positions; measuring  risk; managing  and safeguarding client  assets; determining capital  and reserve  adequacy; and many other activities. In recent years, banks have applied  models to more  complex products  and with more ambitiou

In [9]:
for i, item in enumerate(doc.texts[5:15]):
    print("=" * 80)
    print(f"Item {i}")
    print("Layer :", item.content_layer)
    print("Label :", item.label)
    print("Page  :", item.prov[0].page_no)
    print("Text  :", item.text)
    print("block_id:", item.self_ref)

Item 0
Layer : ContentLayer.BODY
Label : text
Page  : 1
Text  : The expanding use of models in all aspects of banking reflects the extent to which  models can improve business decisions, but models also come with costs. There is the direct  cost of devoting resources to develop and implement models properly.  There are also the potential  indirect costs of relying on models,  such as the possible  adverse  consequences (including financial loss) of decisions based  on models that  are incorrect or misused. Those consequences  should be addressed by active management  of model risk.  [Page  Break]
block_id: #/texts/5
Item 1
Layer : ContentLayer.BODY
Label : footnote
Page  : 1
Text  : -  Unless otherwise indicated, banks refers to national banks and all other institutions for which the Office of the Comptroller of the Currency  is the primary  supervisor, and to bank holding companies,  state  member banks, and all other institutions for which the Federal Reserve Board is the primary sup

In [10]:
doc = result.document

print(type(doc))
print(doc.export_to_dict().keys())

<class 'docling_core.types.doc.document.DoclingDocument'>
dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])


In [11]:
from collections import Counter

counter = Counter()

for item in doc.texts:
    counter[item.label.name] += 1

print(counter)

Counter({'TEXT': 112, 'SECTION_HEADER': 24, 'PAGE_FOOTER': 12, 'FOOTNOTE': 5, 'LIST_ITEM': 5, 'PAGE_HEADER': 1})


In [12]:
from collections import Counter
from docling_core.types.doc import DocItemLabel

# 1. All items Docling labeled as section headers
headers = [t for t in doc.texts if t.label == DocItemLabel.SECTION_HEADER]
print(f"{len(headers)} section headers found:\n")
for h in headers:
    page = h.prov[0].page_no if h.prov else "?"
    print(f"  p{page}: {h.text}")

# 2. Tables check
print(f"\n{len(doc.tables)} tables detected")

# 3. Parse-quality baseline (log this — it's your audit trail)
print(f"\n{len(doc.texts)} text items, labels: {Counter(t.label for t in doc.texts)}")

24 section headers found:

  p1: CONTENTS
  p1: I.  INTRODUCTION
  p1: Board of Governors of the Federal Reserve  System Office of the Comptroller  of the  Currency
  p1: SUPERVISORY GUIDANCE ON MODEL RISK MANAGEMENT
  p2: II. PURPOSE AND SCOPE
  p3: III. OVERVIEW OF MODEL RISK MANAGEMENT
  p5: IV. MODEL DEVELOPMENT, IMPLEMENTATION, AND USE
  p5: Model  Development and Implementation
  p7: Model  Use
  p9: V. MODEL VALIDATION
  p11: Key Elements  of Comprehensive Validation
  p11: 1. Evaluation  of Conceptual  Soundness
  p12: 2. Ongoing  Monitoring
  p13: 3. Outcomes  Analysis
  p15: Validation  of  Vendor  and  Other  Third-Party Products
  p16: VI. GOVERNANCE, POLICIES, AND CONTROLS
  p17: Board  of Directors  and Senior Management
  p17: Policies  and Procedures
  p18: Roles  and Responsibilities
  p19: Internal Audit
  p20: External Resources
  p20: Model Inventory
  p21: Documentation
  p21: VII.  CONCLUSION

1 tables detected

159 text items, labels: Counter({<DocItemLabel.TEXT:

In [13]:
import re

# ---------------------------------------------------------
# Detect section headers like:
# I. INTRODUCTION
# II. PURPOSE AND SCOPE
# III. OVERVIEW OF MODEL RISK MANAGEMENT
# ---------------------------------------------------------

SECTION_RE = re.compile(r"^\s*([IVX]+)\.\s+(.+)")

# ---------------------------------------------------------
# Basic text normalization

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

# ---------------------------------------------------------
# Metadata Extraction
# ---------------------------------------------------------

blocks = []

current_section_id = None
current_section_title = None

DOCUMENT_NAME = "SR11-7"

for item in doc.texts:

    label = item.label.name

    # Ignore page headers and footers
    if label in ["PAGE_HEADER", "PAGE_FOOTER"]:
        continue

    text = normalize_text(item.text)

    page = item.prov[0].page_no if item.prov else None

    # Update current section whenever a section header is found
    if label == "SECTION_HEADER":

        match = SECTION_RE.match(text)

        if match:
            current_section_id = match.group(1)
            current_section_title = match.group(2)

    blocks.append(
        {
            "document": DOCUMENT_NAME,
            "block_id": item.self_ref,
            "page": page,
            "label": label,
            "section_id": current_section_id,
            "section_title": current_section_title,
            "text": text,
        }
    )

print(f"Total blocks extracted : {len(blocks)}")

sections = sorted(
    {b["section_id"] for b in blocks if b["section_id"] is not None}
)

print("Sections Found :", sections)

Total blocks extracted : 146
Sections Found : ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII']


In [14]:
blocks[:5]

[{'document': 'SR11-7',
  'block_id': '#/texts/1',
  'page': 1,
  'label': 'TEXT',
  'section_id': None,
  'section_title': None,
  'text': 'April 4, 2011'},
 {'document': 'SR11-7',
  'block_id': '#/texts/2',
  'page': 1,
  'label': 'SECTION_HEADER',
  'section_id': None,
  'section_title': None,
  'text': 'CONTENTS'},
 {'document': 'SR11-7',
  'block_id': '#/texts/3',
  'page': 1,
  'label': 'SECTION_HEADER',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': 'I. INTRODUCTION'},
 {'document': 'SR11-7',
  'block_id': '#/texts/4',
  'page': 1,
  'label': 'TEXT',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': 'Banks rely heavily on quantitative analysis and models in most aspects of financial decision making. [Fotnote 1 They routinely use models for a broad range of activities, including underwriting credits; valuing exposures, instruments, and positions; measuring risk; managing and safeguarding client assets; determining capital and reserve adequacy; a

In [15]:
for block in blocks:
    if block["text"].strip() == "":
        print(block)

In [16]:
for block in blocks:
    if len(block["text"]) < 5:
        print(block)

{'document': 'SR11-7', 'block_id': '#/texts/10', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '1'}
{'document': 'SR11-7', 'block_id': '#/texts/11', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '2'}
{'document': 'SR11-7', 'block_id': '#/texts/12', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '3'}
{'document': 'SR11-7', 'block_id': '#/texts/13', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '5'}
{'document': 'SR11-7', 'block_id': '#/texts/14', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '9'}
{'document': 'SR11-7', 'block_id': '#/texts/15', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '16'}
{'document': 'SR11-7', 'block_id': '#/texts/16', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': '21'}


In [17]:
import re

for block in blocks:
    if re.match(r"^Page\s+\d+$", block["text"]):
        print(block)

{'document': 'SR11-7', 'block_id': '#/texts/7', 'page': 1, 'label': 'TEXT', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': 'Page 1'}
{'document': 'SR11-7', 'block_id': '#/texts/22', 'page': 2, 'label': 'TEXT', 'section_id': 'II', 'section_title': 'PURPOSE AND SCOPE', 'text': 'Page 2'}
{'document': 'SR11-7', 'block_id': '#/texts/29', 'page': 3, 'label': 'TEXT', 'section_id': 'III', 'section_title': 'OVERVIEW OF MODEL RISK MANAGEMENT', 'text': 'Page 3'}
{'document': 'SR11-7', 'block_id': '#/texts/35', 'page': 4, 'label': 'TEXT', 'section_id': 'III', 'section_title': 'OVERVIEW OF MODEL RISK MANAGEMENT', 'text': 'Page 4'}
{'document': 'SR11-7', 'block_id': '#/texts/44', 'page': 5, 'label': 'TEXT', 'section_id': 'IV', 'section_title': 'MODEL DEVELOPMENT, IMPLEMENTATION, AND USE', 'text': 'Page 5'}
{'document': 'SR11-7', 'block_id': '#/texts/49', 'page': 6, 'label': 'TEXT', 'section_id': 'IV', 'section_title': 'MODEL DEVELOPMENT, IMPLEMENTATION, AND USE', 'text': 'Page 6'}
{'doc

In [18]:
for block in blocks:
    if block["label"] == "SECTION_HEADER":
        print(
            block["page"],
            block["section_id"],
            block["section_title"],
            "->",
            block["text"]
        )

1 None None -> CONTENTS
1 I INTRODUCTION -> I. INTRODUCTION
1 I INTRODUCTION -> Board of Governors of the Federal Reserve System Office of the Comptroller of the Currency
1 I INTRODUCTION -> SUPERVISORY GUIDANCE ON MODEL RISK MANAGEMENT
2 II PURPOSE AND SCOPE -> II. PURPOSE AND SCOPE
3 III OVERVIEW OF MODEL RISK MANAGEMENT -> III. OVERVIEW OF MODEL RISK MANAGEMENT
5 IV MODEL DEVELOPMENT, IMPLEMENTATION, AND USE -> IV. MODEL DEVELOPMENT, IMPLEMENTATION, AND USE
5 IV MODEL DEVELOPMENT, IMPLEMENTATION, AND USE -> Model Development and Implementation
7 IV MODEL DEVELOPMENT, IMPLEMENTATION, AND USE -> Model Use
9 V MODEL VALIDATION -> V. MODEL VALIDATION
11 V MODEL VALIDATION -> Key Elements of Comprehensive Validation
11 V MODEL VALIDATION -> 1. Evaluation of Conceptual Soundness
12 V MODEL VALIDATION -> 2. Ongoing Monitoring
13 V MODEL VALIDATION -> 3. Outcomes Analysis
15 V MODEL VALIDATION -> Validation of Vendor and Other Third-Party Products
16 VI GOVERNANCE, POLICIES, AND CONTROLS ->

In [19]:
for block in blocks:
    if block["section_id"] is None:
        print(block)

{'document': 'SR11-7', 'block_id': '#/texts/1', 'page': 1, 'label': 'TEXT', 'section_id': None, 'section_title': None, 'text': 'April 4, 2011'}
{'document': 'SR11-7', 'block_id': '#/texts/2', 'page': 1, 'label': 'SECTION_HEADER', 'section_id': None, 'section_title': None, 'text': 'CONTENTS'}


In [20]:
from collections import Counter

label_counts = Counter()
for block in blocks:
    label_counts[block["label"]] += 1

print(label_counts)

Counter({'TEXT': 112, 'SECTION_HEADER': 24, 'FOOTNOTE': 5, 'LIST_ITEM': 5})


In [21]:
from collections import Counter

section_counts = Counter()
for block in blocks:
    section_counts[block["section_id"]] += 1

print(section_counts)

Counter({'V': 52, 'VI': 30, 'IV': 24, 'III': 16, 'I': 15, 'II': 5, None: 2, 'VII': 2})


In [22]:
import re

class DocumentCleaner:

    def __init__(self):
        self.page_number_pattern = re.compile(r"^\d+$")
        self.bullet_pattern = re.compile(r"^[•·\-]+$")
        self.document_titles = {
            "SUPERVISORY GUIDANCE ON MODEL RISK MANAGEMENT",
            "Board of Governors of the Federal Reserve System Office of the Comptroller of the Currency"
        }

    def _is_empty(self, text):
        return text.strip() == ""

    def _is_page_number(self, text):
        return bool(self.page_number_pattern.match(text))

    def _is_bullet(self, text):
        return bool(self.bullet_pattern.match(text))

    def _is_document_title(self, text):
        return text.strip() in self.document_titles

    def clean(self, blocks):

        clean_blocks = []

        for block in blocks:

            text = block["text"]

            if self._is_empty(text):
                continue

            if self._is_page_number(text):
                continue

            if self._is_bullet(text):
                continue

            if self._is_document_title(text):
                continue

            clean_blocks.append(block)

        return clean_blocks

In [23]:
cleaner= DocumentCleaner()

In [24]:
sections= cleaner.clean(blocks)

In [25]:
len(sections)

137

In [26]:
from collections import Counter

label_counts = Counter()
for block in sections:
    label_counts[block["label"]] += 1

print(label_counts)

Counter({'TEXT': 105, 'SECTION_HEADER': 22, 'FOOTNOTE': 5, 'LIST_ITEM': 5})


In [27]:
removed_blocks = [i for i in blocks if i not in sections]
removed_blocks

[{'document': 'SR11-7',
  'block_id': '#/texts/8',
  'page': 1,
  'label': 'SECTION_HEADER',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': 'Board of Governors of the Federal Reserve System Office of the Comptroller of the Currency'},
 {'document': 'SR11-7',
  'block_id': '#/texts/9',
  'page': 1,
  'label': 'SECTION_HEADER',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': 'SUPERVISORY GUIDANCE ON MODEL RISK MANAGEMENT'},
 {'document': 'SR11-7',
  'block_id': '#/texts/10',
  'page': 1,
  'label': 'TEXT',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': '1'},
 {'document': 'SR11-7',
  'block_id': '#/texts/11',
  'page': 1,
  'label': 'TEXT',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': '2'},
 {'document': 'SR11-7',
  'block_id': '#/texts/12',
  'page': 1,
  'label': 'TEXT',
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'text': '3'},
 {'document': 'SR11-7',
  'block_id': '#/texts/13',
  'page': 1,
  'l

In [28]:
print(type(sections))          # replace `sections` with your actual variable name
print(len(sections))
print(sections[0])             # what does one record look like?

<class 'list'>
137
{'document': 'SR11-7', 'block_id': '#/texts/1', 'page': 1, 'label': 'TEXT', 'section_id': None, 'section_title': None, 'text': 'April 4, 2011'}


In [29]:
for b in sections:
    if b["label"] == "SECTION_HEADER":
        print(repr(b["text"]), "->", b["section_id"])

'CONTENTS' -> None
'I. INTRODUCTION' -> I
'II. PURPOSE AND SCOPE' -> II
'III. OVERVIEW OF MODEL RISK MANAGEMENT' -> III
'IV. MODEL DEVELOPMENT, IMPLEMENTATION, AND USE' -> IV
'Model Development and Implementation' -> IV
'Model Use' -> IV
'V. MODEL VALIDATION' -> V
'Key Elements of Comprehensive Validation' -> V
'1. Evaluation of Conceptual Soundness' -> V
'2. Ongoing Monitoring' -> V
'3. Outcomes Analysis' -> V
'Validation of Vendor and Other Third-Party Products' -> V
'VI. GOVERNANCE, POLICIES, AND CONTROLS' -> VI
'Board of Directors and Senior Management' -> VI
'Policies and Procedures' -> VI
'Roles and Responsibilities' -> VI
'Internal Audit' -> VI
'External Resources' -> VI
'Model Inventory' -> VI
'Documentation' -> VI
'VII. CONCLUSION' -> VII


In [30]:
import re

ROMAN = re.compile(r"^([IVX]+)\.\s+(.*)$")     # "V. MODEL VALIDATION"
NUMBERED = re.compile(r"^(\d+)\.\s+(.*)$")     # "1. Evaluation of Conceptual Soundness"

cur_roman, cur_roman_title = None, None
cur_sub, cur_sub_title = None, None
sub_counter = 0

for b in sections:
    txt = b["text"].strip()

    if b["label"] == "SECTION_HEADER":
        m = ROMAN.match(txt)
        if m:                                   # top-level heading
            cur_roman, cur_roman_title = m.group(1), m.group(2)
            cur_sub, cur_sub_title, sub_counter = None, None, 0
        elif txt.upper() == txt and len(txt) < 20:
            continue                            # 'CONTENTS' etc.
        else:                                   # sub-heading
            sub_counter += 1
            n = NUMBERED.match(txt)
            cur_sub = f"{sub_counter}"
            cur_sub_title = n.group(2) if n else txt

    # stamp EVERY block (headers included) with where it sits
    b["section_id"] = f"{cur_roman}.{cur_sub}" if cur_sub else cur_roman
    b["section_title"] = cur_sub_title or cur_roman_title

ids = [b["section_id"] for b in sections if b["section_id"]]
print("distinct sections:", len(set(ids)))
for sid in dict.fromkeys(ids):
    print(" ", sid)

distinct sections: 21
  I
  II
  III
  IV
  IV.1
  IV.2
  V
  V.1
  V.2
  V.3
  V.4
  V.5
  VI
  VI.1
  VI.2
  VI.3
  VI.4
  VI.5
  VI.6
  VI.7
  VII


In [31]:
for b in sections:
    if b["label"] == "SECTION_HEADER":
        print(b)

{'document': 'SR11-7', 'block_id': '#/texts/2', 'page': 1, 'label': 'SECTION_HEADER', 'section_id': None, 'section_title': None, 'text': 'CONTENTS'}
{'document': 'SR11-7', 'block_id': '#/texts/3', 'page': 1, 'label': 'SECTION_HEADER', 'section_id': 'I', 'section_title': 'INTRODUCTION', 'text': 'I. INTRODUCTION'}
{'document': 'SR11-7', 'block_id': '#/texts/18', 'page': 2, 'label': 'SECTION_HEADER', 'section_id': 'II', 'section_title': 'PURPOSE AND SCOPE', 'text': 'II. PURPOSE AND SCOPE'}
{'document': 'SR11-7', 'block_id': '#/texts/23', 'page': 3, 'label': 'SECTION_HEADER', 'section_id': 'III', 'section_title': 'OVERVIEW OF MODEL RISK MANAGEMENT', 'text': 'III. OVERVIEW OF MODEL RISK MANAGEMENT'}
{'document': 'SR11-7', 'block_id': '#/texts/39', 'page': 5, 'label': 'SECTION_HEADER', 'section_id': 'IV', 'section_title': 'MODEL DEVELOPMENT, IMPLEMENTATION, AND USE', 'text': 'IV. MODEL DEVELOPMENT, IMPLEMENTATION, AND USE'}
{'document': 'SR11-7', 'block_id': '#/texts/41', 'page': 5, 'label':

In [32]:
grouped = {}

for block in sections:

    sid = block["section_id"]

    if sid is None:
        continue

    if sid not in grouped:

        grouped[sid] = {
            "document": block["document"],
            "section_id": sid,
            "section_title": block["section_title"],
            "page":block["page"],
            "text": block["text"]
        }

    else:

        grouped[sid]["text"] += "\n\n" + block["text"]

sr_sections = list(grouped.values())

print(f"Total Sections: {len(sr_sections)}")

print(len(sr_sections), "sections")
print(sum(len(s["text"]) for s in sr_sections), "total characters")

Total Sections: 21
21 sections
66404 total characters


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
)

chunks_a = []

for sec in sr_sections:

    pieces = splitter.split_text(sec["text"])

    for i, piece in enumerate(pieces):

        chunks_a.append({
            "chunk_id": f"{sec['document']}_{sec['section_id']}_c{i}",
            "document": sec["document"],
            "page": sec["page"],
            "section_id": sec["section_id"],
            "section_title": sec["section_title"],
            "strategy": "recursive",
            "text": piece,
        })

print(f"{len(sr_sections)} sections -> {len(chunks_a)} chunks")

21 sections -> 63 chunks


In [34]:
chunks_a[:5]

[{'chunk_id': 'SR11-7_I_c0',
  'document': 'SR11-7',
  'page': 1,
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'strategy': 'recursive',
  'text': 'I. INTRODUCTION\n\nBanks rely heavily on quantitative analysis and models in most aspects of financial decision making. [Fotnote 1 They routinely use models for a broad range of activities, including underwriting credits; valuing exposures, instruments, and positions; measuring risk; managing and safeguarding client assets; determining capital and reserve adequacy; and many other activities. In recent years, banks have applied models to more complex products and with more ambitious scope, such as enterprise-wide risk measurement, while the markets in which they are used have also broadened and changed. Changes in regulation have spurred some of the recent developments, particularly the U.S. regulatory capital rules for market, credit, and operational risk based on the framework developed by the Basel Committee on Banking Supervi

In [35]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

sem_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

chunks_a2 = []

for sec in sr_sections:

    pieces = sem_splitter.split_text(sec["text"])

    for i, piece in enumerate(pieces):

        chunks_a2.append({
            "chunk_id": f"{sec['document']}_{sec['section_id']}_s{i}",
            "document": sec["document"], 
            "page": sec["page"],
            "section_id": sec["section_id"],
            "section_title": sec["section_title"],
            "strategy": "semantic",
            "text": piece,
        })

print(f"{len(sr_sections)} sections -> {len(chunks_a2)} chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1357.65it/s]


21 sections -> 68 chunks


In [36]:
chunks_a2[:1]

[{'chunk_id': 'SR11-7_I_s0',
  'document': 'SR11-7',
  'page': 1,
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'strategy': 'semantic',
  'text': 'I. INTRODUCTION\n\nBanks rely heavily on quantitative analysis and models in most aspects of financial decision making. [Fotnote 1 They routinely use models for a broad range of activities, including underwriting credits; valuing exposures, instruments, and positions; measuring risk; managing and safeguarding client assets; determining capital and reserve adequacy; and many other activities. In recent years, banks have applied models to more complex products and with more ambitious scope, such as enterprise-wide risk measurement, while the markets in which they are used have also broadened and changed. Changes in regulation have spurred some of the recent developments, particularly the U.S. regulatory capital rules for market, credit, and operational risk based on the framework developed by the Basel Committee on Banking Supervis

In [37]:
parents_b = []
children_b = []

for sec in sr_sections:

    parent_id = f"{sec['document']}_{sec['section_id']}_parent"

    parents_b.append({
        "parent_id": parent_id,
        "document": sec["document"],
        "page": sec["page"],
        "section_id": sec["section_id"],
        "section_title": sec["section_title"],
        "strategy": "parentdoc",
        "text": sec["text"],
    })

    paragraphs = [p.strip() for p in sec["text"].split("\n\n") if p.strip()]

    for i, para in enumerate(paragraphs):

        children_b.append({
            "chunk_id": f"{sec['document']}_{sec['section_id']}_b{i}",
            "parent_id": parent_id,
            "document": sec["document"],
            "page": sec["page"],
            "section_id": sec["section_id"],
            "section_title": sec["section_title"],
            "strategy": "parentdoc",
            "text": para,
        })

print(f"{len(parents_b)} parents, {len(children_b)} children")

21 parents, 135 children


In [38]:
parents_b[:1]

[{'parent_id': 'SR11-7_I_parent',
  'document': 'SR11-7',
  'page': 1,
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'strategy': 'parentdoc',
  'text': 'I. INTRODUCTION\n\nBanks rely heavily on quantitative analysis and models in most aspects of financial decision making. [Fotnote 1 They routinely use models for a broad range of activities, including underwriting credits; valuing exposures, instruments, and positions; measuring risk; managing and safeguarding client assets; determining capital and reserve adequacy; and many other activities. In recent years, banks have applied models to more complex products and with more ambitious scope, such as enterprise-wide risk measurement, while the markets in which they are used have also broadened and changed. Changes in regulation have spurred some of the recent developments, particularly the U.S. regulatory capital rules for market, credit, and operational risk based on the framework developed by the Basel Committee on Banking Su

In [39]:
children_b[:1]

[{'chunk_id': 'SR11-7_I_b0',
  'parent_id': 'SR11-7_I_parent',
  'document': 'SR11-7',
  'page': 1,
  'section_id': 'I',
  'section_title': 'INTRODUCTION',
  'strategy': 'parentdoc',
  'text': 'I. INTRODUCTION'}]

In [43]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = QdrantClient(url="http://localhost:6333")

def index(corpus, collection):
    vectors = model.encode(
        [c["text"] for c in corpus],
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    client.recreate_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    client.upsert(
        collection_name=collection,
        points=[PointStruct(id=i, vector=vectors[i].tolist(), payload=corpus[i])
                for i in range(len(corpus))],
    )
    print(f"{collection}: {client.count(collection).count} points")

index(chunks_a,   "reg_recursive_test")
index(chunks_a2,  "reg_semantic_test")
index(children_b, "reg_parentdoc_test")

parent_lookup = {p["parent_id"]: p for p in parents_b}

Batches: 100%|██████████| 2/2 [00:12<00:00,  6.22s/it]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_14936\2652066654.py:14: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


reg_recursive_test: 63 points


Batches: 100%|██████████| 3/3 [00:19<00:00,  6.36s/it]


reg_semantic_test: 68 points


Batches: 100%|██████████| 5/5 [00:13<00:00,  2.77s/it]


reg_parentdoc_test: 135 points


In [44]:
PREFIX = "Represent this sentence for searching relevant passages: "
question = "What are the key elements of model validation ?"
qvec = model.encode(PREFIX + question, normalize_embeddings=True).tolist()

for coll in ["reg_recursive_test", "reg_semantic_test", "reg_parentdoc_test"]:
    print(f"\n=== {coll} ===")
    for h in client.query_points(coll, query=qvec, limit=3).points:
        print(f"[{h.score:.3f}] §{h.payload['section_id']} {h.payload['section_title']}")
        print("   ", h.payload["text"][:120].replace("\n", " "), "...")

hits = client.query_points("reg_parentdoc_test", query=qvec, limit=5).points
pids = list(dict.fromkeys(h.payload["parent_id"] for h in hits))
print("\nParents to send to LLM:", pids)
print("Total context chars:", sum(len(parent_lookup[p]["text"]) for p in pids))


=== reg_recursive_test ===
[0.830] §V MODEL VALIDATION
    V. MODEL VALIDATION  Model validation is the set of processes and activities intended to verify that models are performi ...
[0.795] §V MODEL VALIDATION
    Effective model validation helps reduce model risk by identifying model errors, corrective actions, and appropriate use. ...
[0.793] §V.1 Key Elements of Comprehensive Validation
    Key Elements of Comprehensive Validation  An effective validation framework should include three core elements:  Evaluat ...

=== reg_semantic_test ===
[0.821] §V MODEL VALIDATION
    V. MODEL VALIDATION  Model validation is the set of processes and activities intended to verify that models are performi ...
[0.813] §III OVERVIEW OF MODEL RISK MANAGEMENT
    Another essential element is a sound model validation process. A third element is governance, which sets an effective fr ...
[0.794] §V MODEL VALIDATION
    Material changes to models should also be subject to validation. It is generally go

In [45]:
from rank_bm25 import BM25Okapi
import re

# --- Registry: the single source of truth for the 3 strategies ---
# Each maps to (Qdrant collection, the in-memory chunk list already indexed there)
STRATEGIES = {
    "recursive":  {"collection": "reg_recursive_test",  "chunks": chunks_a},
    "semantic":   {"collection": "reg_semantic_test",   "chunks": chunks_a2},
    "parentdoc":  {"collection": "reg_parentdoc_test",  "chunks": children_b},
}

def simple_tokenize(text: str) -> list[str]:
    # lowercase, split on non-word chars — good enough for a prototype
    return re.findall(r"\w+", text.lower())

# Build a BM25 index for each strategy, aligned to its chunk list by position
for name, cfg in STRATEGIES.items():
    chunks = cfg["chunks"]
    tokenized_corpus = [simple_tokenize(c["text"]) for c in chunks]
    cfg["bm25"] = BM25Okapi(tokenized_corpus)
    cfg["ids"]  = [c["chunk_id"] for c in chunks]   # position -> chunk_id
    cfg["by_id"] = {c["chunk_id"]: c for c in chunks}  # chunk_id -> chunk (for lookup later)
    print(f"{name:10s}: BM25 built over {len(chunks)} chunks")

recursive : BM25 built over 63 chunks
semantic  : BM25 built over 68 chunks
parentdoc : BM25 built over 135 chunks


In [46]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def dense_search(query, cfg, k=20, framework=None):
    qvec = model.encode(QUERY_PREFIX + query, normalize_embeddings=True)
    qfilter = None
    if framework:
        qfilter = Filter(must=[FieldCondition(key="document", match=MatchValue(value=framework))])
    resp = client.query_points(
        collection_name=cfg["collection"],
        query=qvec.tolist(),        # was query_vector=
        limit=k,
        query_filter=qfilter,
    )
    return [p.payload["chunk_id"] for p in resp.points]   # resp.points, not the raw list

def bm25_search(query, cfg, k=20):
    scores = cfg["bm25"].get_scores(simple_tokenize(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [cfg["ids"][i] for i in top_idx]           # ranked chunk_ids

def rrf(dense_ids, bm25_ids, k=60):
    scores = {}
    for rank, cid in enumerate(dense_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k + rank + 1)
    for rank, cid in enumerate(bm25_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def retrieve(query, strategy, k=5, framework=None):
    cfg = STRATEGIES[strategy]
    dense_ids = dense_search(query, cfg, k=20, framework=framework)
    bm25_ids  = bm25_search(query, cfg, k=20)
    fused = rrf(dense_ids, bm25_ids)[:k]
    return [cfg["by_id"][cid] for cid in fused]        # full chunk dicts, ranked

In [47]:
q = "what is the governance required around models?"
for strat in STRATEGIES:
    print(f"\n=== {strat} ===")
    for c in retrieve(q, strat, k=3):
        print(f"  [{c['section_id']}] {c['text'][:250]}...")


=== recursive ===
  [VI] VI. GOVERNANCE, POLICIES, AND CONTROLS

Developing and maintaining strong governance, policies, and controls over the model risk management framework is fundamentally important to its effectiveness. Even if model development, implementation, use, and...
  [VI.1] Board of Directors and Senior Management

Model risk governance is provided at the highest level by the board of directors and senior management when they establish a bank-wide approach to model risk management. As part of their overall responsibilit...
  [I] The expanding use of models in all aspects of banking reflects the extent to which models can improve business decisions, but models also come with costs. There is the direct cost of devoting resources to develop and implement models properly. There ...

=== semantic ===
  [VI] VI. GOVERNANCE, POLICIES, AND CONTROLS

Developing and maintaining strong governance, policies, and controls over the model risk management framework is fundamentally impor

In [48]:
def compare_arms(query, strategy, k=5):
    cfg = STRATEGIES[strategy]
    dense = dense_search(query, cfg, k=k)
    bm25  = bm25_search(query, cfg, k=k)
    hybrid = [c["chunk_id"] for c in retrieve(query, strategy, k=k)]
    rows = []
    for i in range(k):
        rows.append((
            i+1,
            dense[i]  if i < len(dense)  else "-",
            bm25[i]   if i < len(bm25)   else "-",
            hybrid[i] if i < len(hybrid) else "-",
        ))
    print(f"\nQ: {query}   (strategy={strategy})")
    print(f"{'rank':<5}{'DENSE':<22}{'BM25':<22}{'HYBRID':<22}")
    for r in rows:
        print(f"{r[0]:<5}{r[1]:<22}{r[2]:<22}{r[3]:<22}")

# Two contrasting queries on ONE strategy first (recursive), to see the arms diverge
compare_arms("what does the guidance say about ongoing monitoring of models", "recursive")  # paraphrase-ish
compare_arms("SR 11-7 outcomes analysis benchmarking", "recursive")                          # term-heavy


Q: what does the guidance say about ongoing monitoring of models   (strategy=recursive)
rank DENSE                 BM25                  HYBRID                
1    SR11-7_V.3_c0         SR11-7_V.3_c0         SR11-7_V.3_c0         
2    SR11-7_V.3_c2         SR11-7_V.3_c3         SR11-7_V.3_c2         
3    SR11-7_II_c1          SR11-7_V.3_c2         SR11-7_II_c1          
4    SR11-7_V_c4           SR11-7_II_c1          SR11-7_V.3_c3         
5    SR11-7_VII_c0         SR11-7_III_c4         SR11-7_V_c3           

Q: SR 11-7 outcomes analysis benchmarking   (strategy=recursive)
rank DENSE                 BM25                  HYBRID                
1    SR11-7_V.4_c0         SR11-7_V.1_c0         SR11-7_V.4_c0         
2    SR11-7_V.4_c1         SR11-7_V.5_c1         SR11-7_V.1_c0         
3    SR11-7_V.4_c4         SR11-7_II_c2          SR11-7_V.4_c1         
4    SR11-7_V.3_c3         SR11-7_IV.2_c2        SR11-7_V.4_c4         
5    SR11-7_V.4_c2         SR11-7_V.4_c0         SR11

In [49]:
from sentence_transformers import CrossEncoder

# Small, CPU-practical cross-encoder. First run downloads ~80MB.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_rerank(query, strategy, k_retrieve=20, k_final=5):
    cfg = STRATEGIES[strategy]

    # 1. Wide net: hybrid retrieve 20 candidates (reuse Stage 5)
    candidates = retrieve(query, strategy, k=k_retrieve)   # list of chunk dicts

    # 2. Score each (query, chunk_text) pair jointly
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)                        # higher = more relevant

    # 3. Sort candidates by score, keep top-k_final
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c for c, s in ranked[:k_final]]

C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Administrator\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1765.60it/s]


In [50]:
q = "What is required for effective model validation?"

for strat in STRATEGIES:
    hybrid = retrieve(q, strat, k=5)
    rerank = retrieve_rerank(q, strat)

    print(f"\n=== {strat} ===")
    print("=== HYBRID (before rerank) ===")
    for i, c in enumerate(hybrid, 1):
        print(f"{i}  {c['chunk_id']:<18} {c['text'][:100]}")

    print("\n=== RERANKED (after cross-encoder) ===")
    for i, c in enumerate(rerank, 1):
        print(f"{i}  {c['chunk_id']:<18} {c['text'][:100]}")


=== recursive ===
=== HYBRID (before rerank) ===
1  SR11-7_VI.7_c1     Documentation takes time and effort, and model developers and users who know the models well may not
2  SR11-7_II_c0       II. PURPOSE AND SCOPE

The purpose of this document is to provide comprehensive guidance for banks o
3  SR11-7_V_c0        V. MODEL VALIDATION

Model validation is the set of processes and activities intended to verify that
4  SR11-7_III_c5      A guiding principle for managing model risk is "effective challenge" of models, that is, critical an
5  SR11-7_V_c4        Validation activities should continue on an ongoing basis after a model goes into use, to track know

=== RERANKED (after cross-encoder) ===
1  SR11-7_V_c0        V. MODEL VALIDATION

Model validation is the set of processes and activities intended to verify that
2  SR11-7_V_c5        Effective model validation helps reduce model risk by identifying model errors, corrective actions, 
3  SR11-7_II_c0       II. PURPOSE AND SCOPE

The 

In [51]:
import requests
import re

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3:1.7b"

SYSTEM_PROMPT = """You are a regulatory assistant. The context contains excerpts from regulatory guidance documents, 
each labelled with its source in the form [DOCUMENT §section]. Answer the question using ONLY the provided context.

Rules:
- Every claim must come from the context. Do not use outside knowledge.
- Cite the source in brackets after each claim, e.g. [SR11-7 §V.3].
- If the answer is not in the context, reply exactly: "Not found in the provided documents."
"""

def build_context(query, strategy, k_final=5):
    """Retrieve + rerank, then deliver PARENTS for strategy B (rerank child, read parent)."""
    chunks = retrieve_rerank(query, strategy, k_final=k_final)
    if strategy == "parentdoc":
        seen, parents = set(), []
        for c in chunks:
            pid = c["parent_id"]
            if pid in seen:
                continue
            seen.add(pid)
            parents.append(parent_lookup[pid])   # swap child -> full parent section
        chunks = parents
    return chunks

def format_context(chunks):
    blocks = []
    for c in chunks:
        sid = c.get("section_id", "?")
        title = c.get("section_title", "")
        blocks.append(f"[{sid}] {title}\n{c['text']}")
    return "\n\n---\n\n".join(blocks)

def answer(query, strategy="recursive", k_final=5):
    chunks = build_context(query, strategy, k_final=k_final)
    context = format_context(chunks)

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
        "stream": False,
        "think": False,                 # Qwen3: turn off reasoning trace
        "options": {"temperature": 0},  # reproducibility — non-negotiable
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=360)
    text = resp.json()["message"]["content"]
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()  # fallback strip
    return text, chunks

In [52]:
for strat in STRATEGIES:
    ans, used = answer("Are data proxies permitted ?", strategy=strat)
    print("ANSWER:\n", ans)
    print("\nCONTEXT USED:")

    for c in used:
        print({
            "Section_id": c["section_id"],
            "Chunk_id": c["chunk_id"],
            "Page": c["page"]
    })

print("\n" + "="*60)
ans2, _ = answer("What is the capital requirement ratio under Basel III?", strategy="semantic")
print("OUT-OF-SCOPE ANSWER:\n", ans2)   # SR 11-7 doesn't cover this -> should refuse

ANSWER:
 Not found in the provided documents.

CONTEXT USED:
{'Section_id': 'IV.1', 'Chunk_id': 'SR11-7_IV.1_c1', 'Page': 5}
{'Section_id': 'V', 'Chunk_id': 'SR11-7_V_c3', 'Page': 9}
{'Section_id': 'V.5', 'Chunk_id': 'SR11-7_V.5_c1', 'Page': 15}
{'Section_id': 'V.5', 'Chunk_id': 'SR11-7_V.5_c0', 'Page': 15}
{'Section_id': 'V.3', 'Chunk_id': 'SR11-7_V.3_c1', 'Page': 12}
ANSWER:
 The context does not explicitly address whether data proxies are permitted. Therefore, the answer is: "Not found in the provided documents."

CONTEXT USED:
{'Section_id': 'IV.1', 'Chunk_id': 'SR11-7_IV.1_s1', 'Page': 5}
{'Section_id': 'V', 'Chunk_id': 'SR11-7_V_s3', 'Page': 9}
{'Section_id': 'III', 'Chunk_id': 'SR11-7_III_s3', 'Page': 3}
{'Section_id': 'V.3', 'Chunk_id': 'SR11-7_V.3_s0', 'Page': 12}
{'Section_id': 'V.5', 'Chunk_id': 'SR11-7_V.5_s0', 'Page': 15}
ANSWER:
 Data proxies are permitted [SR11-7 §V.3].

CONTEXT USED:


KeyError: 'chunk_id'

In [53]:
import json
import time

def load_golden(path=r"C:\Users\Administrator\Desktop\AI-Portfolio\RAG\evaluation\golden_set.jsonl"):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def run_harness(strategy, golden, k_final=5):
    results = []
    for i, item in enumerate(golden, 1):
        t0 = time.time()
        ans, chunks = answer(item["question"], strategy=strategy, k_final=k_final)
        dt = time.time() - t0

        results.append({
            # --- keys RAGAS expects ---
            "question": item["question"],
            "answer": ans,
            "document" : [c["document"] for c in chunks],
            "contexts": [c["text"] for c in chunks],
            "ground_truth": item["reference_answer"],
            # --- extra keys for our custom scoring ---
            "id": item["id"],
            "type": item["type"],
            "expected_sections": item["source_sections"],
            "retrieved_sections": [c.get("section_id") for c in chunks],
            "latency_s": round(dt, 1),
        })
        print(f"{i:2d}/{len(golden)}  {item['id']:6s} ({dt:4.1f}s)  {ans[:70]}")
    return results

golden = load_golden()
results_semantic = run_harness("recursive", golden)

 1/13  sr-01  (24.3s)  The definition of a model provided in SR 11-7 is that a model refers t
 2/13  sr-02  (20.5s)  Model risk refers to the risk that a model may produce inaccurate or m
 3/13  sr-03  (22.2s)  "Effective challenge" in model risk management refers to the critical 
 4/13  sr-04  (21.8s)  The three core elements of a comprehensive model validation framework 
 5/13  sr-05  (19.2s)  Evaluation of conceptual soundness involves assessing the quality of t
 6/13  sr-06  (29.6s)  Outcomes analysis is a component of the validation process that involv
 7/13  sr-07  (20.0s)  Model validation should be performed independently of model developmen
 8/13  sr-08  (18.1s)  The ultimate responsibility for model risk management within a bank is
 9/13  sr-09  (27.7s)  A bank's model inventory should contain the purpose and products for w
10/13  sr-10  (20.9s)  Vendor and other third-party models should be validated following the 
11/13  sr-11  (20.3s)  Effective challenge applies across bo

In [54]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

judge_llm = LangchainLLMWrapper(ChatOllama(model="qwen2.5:3b-instruct", temperature=0, format="json"))
judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5"))

def to_ragas(results):
    return EvaluationDataset.from_list([{
        "user_input": r["question"],
        "retrieved_contexts": r["contexts"],
        "response": r["answer"],
        "reference": r["ground_truth"],
    } for r in results])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4035.85it/s]


In [57]:
from ragas import RunConfig

answerable = [r for r in results_semantic if r["type"] != "out_of_scope"]

result = evaluate(
    dataset=to_ragas(answerable),
    metrics=[context_recall, faithfulness],
    llm=judge_llm,
    embeddings=judge_emb,
    run_config=RunConfig(timeout=900, max_workers=1),
)

print(result)

Evaluating:   5%|▍         | 1/22 [00:28<09:53, 28.27s/it]Task was destroyed but it is pending!
task: <Task pending name='Task-1520' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-1521' coro=<Kernel.shell_main() running at C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\Administrator\Desktop\AI-Portfolio\RAG\.venv\Lib\site-packages\requests\sessions.py:508: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def __exit__(self, *args: Any) -> None:
Task was destroyed but it is pending!
task: <Task pending name='Task-1521' coro=<Kernel.shell_main() running at C:\Users\Administrator\Desktop\

{'context_recall': 0.9697, 'faithfulness': 0.4773}


In [58]:
df = result.to_pandas()

# add the question id back for readability, then show the per-question scores
df.insert(0, "id", [r["id"] for r in answerable])
df[["id", "user_input", "context_recall", "faithfulness"]]

,id,user_input,context_recall,faithfulness
0,sr-01,How does SR 11-7 define a model?,1.000000,0.00
1,sr-02,What is model risk and what are its two main s...,1.000000,0.00
2,sr-03,What is 'effective challenge' in model risk ma...,1.000000,0.50
3,sr-04,What are the three core elements of a comprehe...,1.000000,1.00
4,sr-05,What does evaluation of conceptual soundness i...,1.000000,1.00
5,sr-06,What is outcomes analysis and how is backtesti...,1.000000,1.00
6,sr-07,Why should model validation be performed indep...,1.000000,1.00
7,sr-08,Who holds ultimate responsibility for model ri...,1.000000,0.75
8,sr-09,What should a bank's model inventory contain?,1.000000,0.00
9,sr-10,How should vendor and other third-party models...,1.000000,0.00


In [59]:
import json

with open("results_recursive.json", "w", encoding="utf-8") as f:
    json.dump(results_semantic, f, indent=2, ensure_ascii=False)

In [66]:
df.to_json("eval_result_recursive.json", orient="records", indent=2)

In [65]:
import os

print(os.getcwd())

C:\Users\Administrator\Desktop\AI-Portfolio\RAG\notebooks\Prototype
